In [2]:
!wget https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv

--2025-09-18 20:50:29--  https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26130198 (25M) [text/plain]
Saving to: ‘combined_dataset.csv’

combined_dataset.cs 100%[===================>]  24.92M  --.-KB/s    in 0.05s   

2025-09-18 20:50:32 (506 MB/s) - ‘combined_dataset.csv’ saved [26130198/26130198]



In [3]:
!pip install pandas numpy pytorch-crf seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6dd2700c28a6bdef0d2763604c255e258b8f5b9d8978e0a7422e5ec6a0c3d98c
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score as sklearn_f1_score, classification_report as sklearn_classification_report
from tensorflow.keras.preprocessing.sequence import pad_sequences
from torchcrf import CRF
from seqeval.metrics import classification_report as ner_classification_report, f1_score as ner_f1_score

In [10]:
class EarlyStopping:
    """Early stopping utility to prevent overfitting"""

    def __init__(self, patience=5, min_delta=0.0001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_score = None
        self.counter = 0
        self.best_weights = None
        self.early_stop = False

    def __call__(self, val_score, model):
        if self.best_score is None:
            self.best_score = val_score
            self.save_checkpoint(model)
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        '''Save model when validation score improves'''
        if self.restore_best_weights:
            self.best_weights = model.state_dict().copy()


In [ ]:
class NERDataProcessor:
    """Handles data loading, preprocessing, and vocabulary building"""

    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.data_df = None
        self.word2idx = None
        self.lemma2idx = None
        self.pos2idx = None
        self.tag2idx = None
        self.idx2tag = None
        self.MAX_LEN = None

    def load_data(self):
        """Load the CSV dataset"""
        self.data_df = pd.read_csv(self.csv_path)
        return self.data_df

    def build_vocabularies(self):
        """Build vocabularies for all features"""
        # Word vocabulary
        unique_words = sorted(list(set(self.data_df["WORD"].str.lower().values)))
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        for i, word in enumerate(unique_words):
            self.word2idx[word] = i + 2

        # Lemma vocabulary
        unique_lemmas = sorted(list(set(self.data_df["LEMMA"].str.lower().values)))
        self.lemma2idx = {"<PAD>": 0, "<UNK>": 1}
        for i, lemma in enumerate(unique_lemmas):
            self.lemma2idx[lemma] = i + 2

        # POS vocabulary
        unique_pos = sorted(list(set(self.data_df["POS_TAG"].values)))
        self.pos2idx = {"<PAD>": 0, "<UNK>": 1}
        for i, pos in enumerate(unique_pos):
            self.pos2idx[pos] = i + 2

        # NER tag vocabulary
        tags = sorted(list(set(self.data_df["NER_TAG"].values)))
        self.tag2idx = {"<PAD>": 0}
        for i, tag in enumerate(tags):
            self.tag2idx[tag] = i + 1
        self.idx2tag = {v: k for k, v in self.tag2idx.items()}

    def prepare_sequences(self):
        """Convert data to sequences and pad them"""
        # Convert to sequences
        sentences = self.data_df.groupby("SENTENCE #").apply(
            lambda group: list(zip(group["WORD"].values, group["LEMMA"].values,
                                 group["POS_TAG"].values, group["NER_TAG"].values)),
            include_groups=False
        ).tolist()

        X_words = [[word for word, lemma, pos, tag in s] for s in sentences]
        X_lemmas = [[lemma for word, lemma, pos, tag in s] for s in sentences]
        X_pos = [[pos for word, lemma, pos, tag in s] for s in sentences]
        y_ner = [[tag for word, lemma, pos, tag in s] for s in sentences]

        # Map to indices
        X_words_idx = [[self.word2idx.get(w.lower(), self.word2idx["<UNK>"]) for w in seq] for seq in X_words]
        X_lemmas_idx = [[self.lemma2idx.get(l.lower(), self.lemma2idx["<UNK>"]) for l in seq] for seq in X_lemmas]
        X_pos_idx = [[self.pos2idx.get(p, self.pos2idx["<UNK>"]) for p in seq] for seq in X_pos]
        y_ner_idx = [[self.tag2idx.get(t, self.tag2idx["<PAD>"]) for t in seq] for seq in y_ner]

        self.MAX_LEN = 128

        # Pad sequences
        X_words_pad = pad_sequences(X_words_idx, maxlen=self.MAX_LEN, padding="post", value=self.word2idx["<PAD>"])
        X_lemmas_pad = pad_sequences(X_lemmas_idx, maxlen=self.MAX_LEN, padding="post", value=self.lemma2idx["<PAD>"])
        X_pos_pad = pad_sequences(X_pos_idx, maxlen=self.MAX_LEN, padding="post", value=self.pos2idx["<PAD>"])
        y_ner_pad = pad_sequences(y_ner_idx, maxlen=self.MAX_LEN, padding="post", value=self.tag2idx["<PAD>"])

        return X_words_pad, X_lemmas_pad, X_pos_pad, y_ner_pad

    def create_splits(self, X_words_pad, X_lemmas_pad, X_pos_pad, y_ner_pad, test_size=0.2, val_size=0.1):
        """Create train/validation/test splits"""
        # First split: 80% train, 20% (temp) validation
        Xw_train, Xw_temp, Xl_train, Xl_temp, Xp_train, Xp_temp, yn_train, yn_temp = train_test_split(
            X_words_pad, X_lemmas_pad, X_pos_pad, y_ner_pad, test_size=0.2, random_state=42
        )

        # Split the 20% temp into 10% val and 10% test (so 50/50 of temp)
        Xw_val, Xw_test, Xl_val, Xl_test, Xp_val, Xp_test, yn_val, yn_test = train_test_split(
            Xw_temp, Xl_temp, Xp_temp, yn_temp, test_size=0.5, random_state=42
        )

        return {
            'train': (Xw_train, Xl_train, Xp_train, yn_train),
            'val': (Xw_val, Xl_val, Xp_val, yn_val),
            'test': (Xw_test, Xl_test, Xp_test, yn_test)
        }


class NEREvaluator:
    """Handles evaluation metrics for NER models"""

    @staticmethod
    def calculate_metrics(y_true_sequences, y_pred_sequences, idx2tag):
        """Calculate both entity-level and token-level F1 scores"""
        # Convert to tag sequences
        y_true_seq = [[idx2tag[idx] for idx in seq] for seq in y_true_sequences]
        y_pred_seq = [[idx2tag[idx] for idx in seq] for seq in y_pred_sequences]

        # Entity-level F1 (seqeval)
        entity_f1 = ner_f1_score(y_true_seq, y_pred_seq)

        # Token-level F1 (sklearn)
        y_true_flat = [tag for seq in y_true_seq for tag in seq]
        y_pred_flat = [tag for seq in y_pred_seq for tag in seq]
        token_f1 = sklearn_f1_score(y_true_flat, y_pred_flat, average='micro')

        return entity_f1, token_f1


# Dataset Classes
class NERDatasetSimple(Dataset):
    """Dataset class for simple NER data (words only)"""
    def __init__(self, words, ner_tags):
        self.words = words
        self.ner_tags = ner_tags

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        return self.words[idx], self.ner_tags[idx]


class NERDatasetPOS(Dataset):
    """Dataset class for NER with words and POS tags"""
    def __init__(self, words, pos_tags, ner_tags):
        self.words = words
        self.pos_tags = pos_tags
        self.ner_tags = ner_tags

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        return self.words[idx], self.pos_tags[idx], self.ner_tags[idx]


class NERDatasetFull(Dataset):
    """Dataset class for full NER data (words, lemmas, POS tags)"""
    def __init__(self, words, lemmas, pos_tags, ner_tags):
        self.words = words
        self.lemmas = lemmas
        self.pos_tags = pos_tags
        self.ner_tags = ner_tags

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        return self.words[idx], self.lemmas[idx], self.pos_tags[idx], self.ner_tags[idx]

In [11]:
# Model 1: BiLSTM Simple (words only) with Early Stopping
class BiLSTMSimpleModel:
    """BiLSTM model with words only"""

    def __init__(self, processor, embedding_dim=200, hidden_dim=128, dropout=0.3, lr=0.001, batch_size=64):
        self.processor = processor
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.best_entity_f1 = 0
        self.best_token_f1 = 0

    def build_model(self):
        """Build the BiLSTM model"""
        class BiLSTMSimple(nn.Module):
            def __init__(self, vocab_size, tag_count, embedding_dim, hidden_dim, dropout, pad_idx):
                super(BiLSTMSimple, self).__init__()
                self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
                self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2,
                                   batch_first=True, bidirectional=True, dropout=dropout)
                self.dropout = nn.Dropout(dropout)
                self.classifier = nn.Linear(hidden_dim * 2, tag_count)

            def forward(self, x):
                embedded = self.embedding(x)
                lstm_out, _ = self.lstm(embedded)
                lstm_out = self.dropout(lstm_out)
                output = self.classifier(lstm_out)
                return output

        self.model = BiLSTMSimple(
            vocab_size=len(self.processor.word2idx),
            tag_count=len(self.processor.tag2idx),
            embedding_dim=self.embedding_dim,
            hidden_dim=self.hidden_dim,
            dropout=self.dropout,
            pad_idx=self.processor.word2idx["<PAD>"]
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.processor.tag2idx["<PAD>"])

    def create_data_loaders(self, splits):
        """Create data loaders"""
        train_dataset = NERDatasetSimple(splits['train'][0], splits['train'][3])
        val_dataset = NERDatasetSimple(splits['val'][0], splits['val'][3])
        test_dataset = NERDatasetSimple(splits['test'][0], splits['test'][3])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=50, patience=7):
        """Train the model with early stopping"""
        print("Training BiLSTM Simple Model...")

        early_stopping = EarlyStopping(patience=patience, min_delta=0.0001)

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                words, tags = batch
                words, tags = words.to(self.device), tags.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(words)
                loss = self.criterion(outputs.view(-1, len(self.processor.tag2idx)), tags.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            # Validation
            entity_f1, token_f1 = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss/len(train_loader):.4f}, "
                  f"Entity F1={entity_f1:.4f}, Token F1={token_f1:.4f}")

            # Track best scores and save best model
            if entity_f1 > self.best_entity_f1:
                self.best_entity_f1 = entity_f1
                self.best_token_f1 = token_f1
                torch.save(self.model.state_dict(), "best_bilstm_simple.pth")

            # Early stopping check
            early_stopping(entity_f1, self.model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

        # Restore best weights if early stopping was used
        if early_stopping.restore_best_weights and early_stopping.best_weights is not None:
            self.model.load_state_dict(early_stopping.best_weights)

    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        y_true_sequences, y_pred_sequences = [], []

        with torch.no_grad():
            for batch in data_loader:
                words, tags = batch
                words, tags = words.to(self.device), tags.to(self.device)

                outputs = self.model(words)
                predictions = torch.argmax(outputs, dim=2)

                mask = (words != self.processor.word2idx["<PAD>"]).cpu()

                for i in range(len(mask)):
                    true_seq = tags[i][mask[i]].cpu().numpy()
                    pred_seq = predictions[i][mask[i]].cpu().numpy()
                    y_true_sequences.append(true_seq)
                    y_pred_sequences.append(pred_seq)

        return NEREvaluator.calculate_metrics(y_true_sequences, y_pred_sequences, self.processor.idx2tag)

    def test(self, test_loader):
        """Test the model with best weights"""
        self.model.load_state_dict(torch.load("best_bilstm_simple.pth"))
        return self.evaluate(test_loader)


# Model 2: BiLSTM + POS with Early Stopping
class BiLSTMPOSModel:
    """BiLSTM model with words and POS tags"""

    def __init__(self, processor, word_emb_dim=200, pos_emb_dim=50,
                 hidden_dim=128, dropout=0.3, lr=0.001, batch_size=64):
        self.processor = processor
        self.word_emb_dim = word_emb_dim
        self.pos_emb_dim = pos_emb_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.best_entity_f1 = 0
        self.best_token_f1 = 0

    def build_model(self):
        """Build the BiLSTM with POS features model"""
        class BiLSTMWithPOS(nn.Module):
            def __init__(self, word_vocab_size, pos_vocab_size, tag_count,
                         word_emb_dim, pos_emb_dim, hidden_dim, dropout,
                         word_pad_idx, pos_pad_idx):
                super(BiLSTMWithPOS, self).__init__()

                self.word_embedding = nn.Embedding(word_vocab_size, word_emb_dim, padding_idx=word_pad_idx)
                self.pos_embedding = nn.Embedding(pos_vocab_size, pos_emb_dim, padding_idx=pos_pad_idx)

                combined_emb_dim = word_emb_dim + pos_emb_dim

                self.lstm = nn.LSTM(combined_emb_dim, hidden_dim, num_layers=2,
                                   bidirectional=True, batch_first=True, dropout=dropout)
                self.dropout = nn.Dropout(dropout)
                self.classifier = nn.Linear(hidden_dim * 2, tag_count)

            def forward(self, words, pos_tags):
                word_emb = self.word_embedding(words)
                pos_emb = self.pos_embedding(pos_tags)

                combined_emb = torch.cat([word_emb, pos_emb], dim=2)
                lstm_out, _ = self.lstm(combined_emb)
                lstm_out = self.dropout(lstm_out)
                output = self.classifier(lstm_out)
                return output

        self.model = BiLSTMWithPOS(
            word_vocab_size=len(self.processor.word2idx),
            pos_vocab_size=len(self.processor.pos2idx),
            tag_count=len(self.processor.tag2idx),
            word_emb_dim=self.word_emb_dim,
            pos_emb_dim=self.pos_emb_dim,
            hidden_dim=self.hidden_dim,
            dropout=self.dropout,
            word_pad_idx=self.processor.word2idx["<PAD>"],
            pos_pad_idx=self.processor.pos2idx["<PAD>"]
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.processor.tag2idx["<PAD>"])

    def create_data_loaders(self, splits):
        """Create data loaders"""
        train_dataset = NERDatasetPOS(splits['train'][0], splits['train'][2], splits['train'][3])
        val_dataset = NERDatasetPOS(splits['val'][0], splits['val'][2], splits['val'][3])
        test_dataset = NERDatasetPOS(splits['test'][0], splits['test'][2], splits['test'][3])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=50, patience=7):
        """Train the model with early stopping"""
        print("Training BiLSTM with POS Features Model...")

        early_stopping = EarlyStopping(patience=patience, min_delta=0.0001)

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                words, pos_tags, tags = batch
                words = words.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(words, pos_tags)
                loss = self.criterion(outputs.view(-1, len(self.processor.tag2idx)), tags.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            # Validation
            entity_f1, token_f1 = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss/len(train_loader):.4f}, "
                  f"Entity F1={entity_f1:.4f}, Token F1={token_f1:.4f}")

            # Track best scores and save best model
            if entity_f1 > self.best_entity_f1:
                self.best_entity_f1 = entity_f1
                self.best_token_f1 = token_f1
                torch.save(self.model.state_dict(), "best_bilstm_pos.pth")

            # Early stopping check
            early_stopping(entity_f1, self.model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

        # Restore best weights if early stopping was used
        if early_stopping.restore_best_weights and early_stopping.best_weights is not None:
            self.model.load_state_dict(early_stopping.best_weights)

    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        y_true_sequences, y_pred_sequences = [], []

        with torch.no_grad():
            for batch in data_loader:
                words, pos_tags, tags = batch
                words = words.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)

                outputs = self.model(words, pos_tags)
                predictions = torch.argmax(outputs, dim=2)

                mask = (words != self.processor.word2idx["<PAD>"]).cpu()

                for i in range(len(mask)):
                    true_seq = tags[i][mask[i]].cpu().numpy()
                    pred_seq = predictions[i][mask[i]].cpu().numpy()
                    y_true_sequences.append(true_seq)
                    y_pred_sequences.append(pred_seq)

        return NEREvaluator.calculate_metrics(y_true_sequences, y_pred_sequences, self.processor.idx2tag)

    def test(self, test_loader):
        """Test the model with best weights"""
        self.model.load_state_dict(torch.load("best_bilstm_pos.pth"))
        return self.evaluate(test_loader)


# Model 3: BiLSTM Full Features with Early Stopping
class BiLSTMFeaturesModel:
    """BiLSTM model with words, lemmas, and POS tags"""

    def __init__(self, processor, word_emb_dim=200, lemma_emb_dim=50, pos_emb_dim=50,
                 hidden_dim=128, dropout=0.3, lr=0.001, batch_size=64):
        self.processor = processor
        self.word_emb_dim = word_emb_dim
        self.lemma_emb_dim = lemma_emb_dim
        self.pos_emb_dim = pos_emb_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.best_entity_f1 = 0
        self.best_token_f1 = 0

    def build_model(self):
        """Build the BiLSTM with features model"""
        class BiLSTMWithFeatures(nn.Module):
            def __init__(self, word_vocab_size, lemma_vocab_size, pos_vocab_size, tag_count,
                         word_emb_dim, lemma_emb_dim, pos_emb_dim, hidden_dim, dropout,
                         word_pad_idx, lemma_pad_idx, pos_pad_idx):
                super(BiLSTMWithFeatures, self).__init__()

                self.word_embedding = nn.Embedding(word_vocab_size, word_emb_dim, padding_idx=word_pad_idx)
                self.lemma_embedding = nn.Embedding(lemma_vocab_size, lemma_emb_dim, padding_idx=lemma_pad_idx)
                self.pos_embedding = nn.Embedding(pos_vocab_size, pos_emb_dim, padding_idx=pos_pad_idx)

                combined_emb_dim = word_emb_dim + lemma_emb_dim + pos_emb_dim

                self.lstm = nn.LSTM(combined_emb_dim, hidden_dim, num_layers=2,
                                   bidirectional=True, batch_first=True, dropout=dropout)
                self.dropout = nn.Dropout(dropout)
                self.classifier = nn.Linear(hidden_dim * 2, tag_count)

            def forward(self, words, lemmas, pos_tags):
                word_emb = self.word_embedding(words)
                lemma_emb = self.lemma_embedding(lemmas)
                pos_emb = self.pos_embedding(pos_tags)

                combined_emb = torch.cat([word_emb, lemma_emb, pos_emb], dim=2)
                lstm_out, _ = self.lstm(combined_emb)
                lstm_out = self.dropout(lstm_out)
                output = self.classifier(lstm_out)
                return output

        self.model = BiLSTMWithFeatures(
            word_vocab_size=len(self.processor.word2idx),
            lemma_vocab_size=len(self.processor.lemma2idx),
            pos_vocab_size=len(self.processor.pos2idx),
            tag_count=len(self.processor.tag2idx),
            word_emb_dim=self.word_emb_dim,
            lemma_emb_dim=self.lemma_emb_dim,
            pos_emb_dim=self.pos_emb_dim,
            hidden_dim=self.hidden_dim,
            dropout=self.dropout,
            word_pad_idx=self.processor.word2idx["<PAD>"],
            lemma_pad_idx=self.processor.lemma2idx["<PAD>"],
            pos_pad_idx=self.processor.pos2idx["<PAD>"]
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.processor.tag2idx["<PAD>"])

    def create_data_loaders(self, splits):
        """Create data loaders"""
        train_dataset = NERDatasetFull(*splits['train'])
        val_dataset = NERDatasetFull(*splits['val'])
        test_dataset = NERDatasetFull(*splits['test'])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=50, patience=7):
        """Train the model with early stopping"""
        print("Training BiLSTM with Full Features Model...")

        early_stopping = EarlyStopping(patience=patience, min_delta=0.0001)

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                words, lemmas, pos_tags, tags = batch
                words = words.to(self.device)
                lemmas = lemmas.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(words, lemmas, pos_tags)
                loss = self.criterion(outputs.view(-1, len(self.processor.tag2idx)), tags.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            # Validation
            entity_f1, token_f1 = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss/len(train_loader):.4f}, "
                f"Entity F1={entity_f1:.4f}, Token F1={token_f1:.4f}")

            # Track best scores and save best model
            if entity_f1 > self.best_entity_f1:
                self.best_entity_f1 = entity_f1
                self.best_token_f1 = token_f1
                torch.save(self.model.state_dict(), "best_bilstm_features.pth")

            # Early stopping check
            early_stopping(entity_f1, self.model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

        # Restore best weights if early stopping was used
        if early_stopping.restore_best_weights and early_stopping.best_weights is not None:
            self.model.load_state_dict(early_stopping.best_weights)


    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        y_true_sequences, y_pred_sequences = [], []

        with torch.no_grad():
            for batch in data_loader:
                words, lemmas, pos_tags, tags = batch
                words = words.to(self.device)
                lemmas = lemmas.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)

                outputs = self.model(words, lemmas, pos_tags)
                predictions = torch.argmax(outputs, dim=2)

                mask = (words != self.processor.word2idx["<PAD>"]).cpu()

                for i in range(len(mask)):
                    true_seq = tags[i][mask[i]].cpu().numpy()
                    pred_seq = predictions[i][mask[i]].cpu().numpy()
                    y_true_sequences.append(true_seq)
                    y_pred_sequences.append(pred_seq)

        return NEREvaluator.calculate_metrics(y_true_sequences, y_pred_sequences, self.processor.idx2tag)

    def test(self, test_loader):
        """Test the model with best weights"""
        self.model.load_state_dict(torch.load("best_bilstm_features.pth"))
        return self.evaluate(test_loader)


# Model 4: BiLSTM-CRF Simple with Early Stopping
class BiLSTMCRFSimpleModel:
    """BiLSTM-CRF model with words only"""

    def __init__(self, processor, embedding_dim=200, hidden_dim=128, dropout=0.3, lr=0.001, batch_size=64):
        self.processor = processor
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.best_entity_f1 = 0
        self.best_token_f1 = 0

    def build_model(self):
        """Build the BiLSTM-CRF model"""
        class BiLSTMCRFSimple(nn.Module):
            def __init__(self, vocab_size, tag_count, embedding_dim, hidden_dim, dropout, pad_idx):
                super(BiLSTMCRFSimple, self).__init__()
                self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
                self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2,
                                   batch_first=True, bidirectional=True, dropout=dropout)
                self.dropout = nn.Dropout(dropout)
                self.hidden2tag = nn.Linear(hidden_dim * 2, tag_count)
                self.crf = CRF(tag_count, batch_first=True)

            def forward(self, x, tags=None, mask=None):
                embedded = self.embedding(x)
                lstm_out, _ = self.lstm(embedded)
                lstm_out = self.dropout(lstm_out)
                logits = self.hidden2tag(lstm_out)

                if tags is not None:
                    loss = -self.crf(logits, tags, mask=mask, reduction="mean")
                    return {"loss": loss}
                else:
                    predictions = self.crf.decode(logits, mask=mask)
                    return {"predictions": predictions}

        self.model = BiLSTMCRFSimple(
            vocab_size=len(self.processor.word2idx),
            tag_count=len(self.processor.tag2idx),
            embedding_dim=self.embedding_dim,
            hidden_dim=self.hidden_dim,
            dropout=self.dropout,
            pad_idx=self.processor.word2idx["<PAD>"]
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def create_data_loaders(self, splits):
        """Create data loaders"""
        train_dataset = NERDatasetSimple(splits['train'][0], splits['train'][3])
        val_dataset = NERDatasetSimple(splits['val'][0], splits['val'][3])
        test_dataset = NERDatasetSimple(splits['test'][0], splits['test'][3])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=50, patience=7):
        """Train the model with early stopping"""
        print("Training BiLSTM-CRF Simple Model...")

        early_stopping = EarlyStopping(patience=patience, min_delta=0.0001)

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                words, tags = batch
                words, tags = words.to(self.device), tags.to(self.device)
                mask = (words != self.processor.word2idx["<PAD>"]).to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(words, tags=tags, mask=mask)
                loss = outputs["loss"]
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            # Validation
            entity_f1, token_f1 = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss/len(train_loader):.4f}, "
                  f"Entity F1={entity_f1:.4f}, Token F1={token_f1:.4f}")

            # Track best scores and save best model
            if entity_f1 > self.best_entity_f1:
                self.best_entity_f1 = entity_f1
                self.best_token_f1 = token_f1
                torch.save(self.model.state_dict(), "best_bilstm_crf_simple.pth")

            # Early stopping check
            early_stopping(entity_f1, self.model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

        # Restore best weights if early stopping was used
        if early_stopping.restore_best_weights and early_stopping.best_weights is not None:
            self.model.load_state_dict(early_stopping.best_weights)

    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        y_true_sequences, y_pred_sequences = [], []

        with torch.no_grad():
            for batch in data_loader:
                words, tags = batch
                words, tags = words.to(self.device), tags.to(self.device)
                mask = (words != self.processor.word2idx["<PAD>"]).to(self.device)

                outputs = self.model(words, mask=mask)
                predictions = outputs["predictions"]

                for i, pred_seq in enumerate(predictions):
                    true_seq = tags[i][mask[i]].cpu().numpy()
                    y_true_sequences.append(true_seq)
                    y_pred_sequences.append(pred_seq)

        return NEREvaluator.calculate_metrics(y_true_sequences, y_pred_sequences, self.processor.idx2tag)

    def test(self, test_loader):
        """Test the model with best weights"""
        self.model.load_state_dict(torch.load("best_bilstm_crf_simple.pth"))
        return self.evaluate(test_loader)


# Model 5: BiLSTM-CRF + POS with Early Stopping
class BiLSTMCRFPOSModel:
    """BiLSTM-CRF model with words and POS tags"""

    def __init__(self, processor, word_emb_dim=200, pos_emb_dim=50,
                 hidden_dim=128, dropout=0.3, lr=0.001, batch_size=64):
        self.processor = processor
        self.word_emb_dim = word_emb_dim
        self.pos_emb_dim = pos_emb_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.best_entity_f1 = 0
        self.best_token_f1 = 0

    def build_model(self):
        """Build the BiLSTM-CRF with POS features model"""
        class BiLSTMCRFWithPOS(nn.Module):
            def __init__(self, word_vocab_size, pos_vocab_size, tag_count,
                         word_emb_dim, pos_emb_dim, hidden_dim, dropout,
                         word_pad_idx, pos_pad_idx):
                super(BiLSTMCRFWithPOS, self).__init__()

                self.word_embedding = nn.Embedding(word_vocab_size, word_emb_dim, padding_idx=word_pad_idx)
                self.pos_embedding = nn.Embedding(pos_vocab_size, pos_emb_dim, padding_idx=pos_pad_idx)

                combined_emb_dim = word_emb_dim + pos_emb_dim

                self.lstm = nn.LSTM(combined_emb_dim, hidden_dim, num_layers=2,
                                   bidirectional=True, batch_first=True, dropout=dropout)
                self.dropout = nn.Dropout(dropout)
                self.hidden2tag = nn.Linear(hidden_dim * 2, tag_count)
                self.crf = CRF(tag_count, batch_first=True)

            def forward(self, words, pos_tags, tags=None, mask=None):
                word_emb = self.word_embedding(words)
                pos_emb = self.pos_embedding(pos_tags)

                combined_emb = torch.cat([word_emb, pos_emb], dim=2)
                lstm_out, _ = self.lstm(combined_emb)
                lstm_out = self.dropout(lstm_out)
                logits = self.hidden2tag(lstm_out)

                if tags is not None:
                    loss = -self.crf(logits, tags, mask=mask, reduction="mean")
                    return {"loss": loss}
                else:
                    predictions = self.crf.decode(logits, mask=mask)
                    return {"predictions": predictions}

        self.model = BiLSTMCRFWithPOS(
            word_vocab_size=len(self.processor.word2idx),
            pos_vocab_size=len(self.processor.pos2idx),
            tag_count=len(self.processor.tag2idx),
            word_emb_dim=self.word_emb_dim,
            pos_emb_dim=self.pos_emb_dim,
            hidden_dim=self.hidden_dim,
            dropout=self.dropout,
            word_pad_idx=self.processor.word2idx["<PAD>"],
            pos_pad_idx=self.processor.pos2idx["<PAD>"]
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def create_data_loaders(self, splits):
        """Create data loaders"""
        train_dataset = NERDatasetPOS(splits['train'][0], splits['train'][2], splits['train'][3])
        val_dataset = NERDatasetPOS(splits['val'][0], splits['val'][2], splits['val'][3])
        test_dataset = NERDatasetPOS(splits['test'][0], splits['test'][2], splits['test'][3])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=50, patience=7):
        """Train the model with early stopping"""
        print("Training BiLSTM-CRF with POS Features Model...")

        early_stopping = EarlyStopping(patience=patience, min_delta=0.0001)

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                words, pos_tags, tags = batch
                words = words.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)
                mask = (words != self.processor.word2idx["<PAD>"]).to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(words, pos_tags, tags=tags, mask=mask)
                loss = outputs["loss"]
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            # Validation
            entity_f1, token_f1 = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss/len(train_loader):.4f}, "
                  f"Entity F1={entity_f1:.4f}, Token F1={token_f1:.4f}")

            # Track best scores and save best model
            if entity_f1 > self.best_entity_f1:
                self.best_entity_f1 = entity_f1
                self.best_token_f1 = token_f1
                torch.save(self.model.state_dict(), "best_bilstm_crf_pos.pth")

            # Early stopping check
            early_stopping(entity_f1, self.model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

        # Restore best weights if early stopping was used
        if early_stopping.restore_best_weights and early_stopping.best_weights is not None:
            self.model.load_state_dict(early_stopping.best_weights)

    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        y_true_sequences, y_pred_sequences = [], []

        with torch.no_grad():
            for batch in data_loader:
                words, pos_tags, tags = batch
                words = words.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)
                mask = (words != self.processor.word2idx["<PAD>"]).to(self.device)

                outputs = self.model(words, pos_tags, mask=mask)
                predictions = outputs["predictions"]

                for i, pred_seq in enumerate(predictions):
                    true_seq = tags[i][mask[i]].cpu().numpy()
                    y_true_sequences.append(true_seq)
                    y_pred_sequences.append(pred_seq)

        return NEREvaluator.calculate_metrics(y_true_sequences, y_pred_sequences, self.processor.idx2tag)

    def test(self, test_loader):
        """Test the model with best weights"""
        self.model.load_state_dict(torch.load("best_bilstm_crf_pos.pth"))
        return self.evaluate(test_loader)


# Model 6: BiLSTM-CRF Full Features with Early Stopping
class BiLSTMCRFFeaturesModel:
    """BiLSTM-CRF model with words, lemmas, and POS tags"""

    def __init__(self, processor, word_emb_dim=200, lemma_emb_dim=50, pos_emb_dim=50,
                 hidden_dim=128, dropout=0.3, lr=0.001, batch_size=64):
        self.processor = processor
        self.word_emb_dim = word_emb_dim
        self.lemma_emb_dim = lemma_emb_dim
        self.pos_emb_dim = pos_emb_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.best_entity_f1 = 0
        self.best_token_f1 = 0

    def build_model(self):
        """Build the BiLSTM-CRF with features model"""
        class BiLSTMCRFWithFeatures(nn.Module):
            def __init__(self, word_vocab_size, lemma_vocab_size, pos_vocab_size, tag_count,
                         word_emb_dim, lemma_emb_dim, pos_emb_dim, hidden_dim, dropout,
                         word_pad_idx, lemma_pad_idx, pos_pad_idx):
                super(BiLSTMCRFWithFeatures, self).__init__()

                self.word_embedding = nn.Embedding(word_vocab_size, word_emb_dim, padding_idx=word_pad_idx)
                self.lemma_embedding = nn.Embedding(lemma_vocab_size, lemma_emb_dim, padding_idx=lemma_pad_idx)
                self.pos_embedding = nn.Embedding(pos_vocab_size, pos_emb_dim, padding_idx=pos_pad_idx)

                combined_emb_dim = word_emb_dim + lemma_emb_dim + pos_emb_dim

                self.lstm = nn.LSTM(combined_emb_dim, hidden_dim, num_layers=2,
                                   bidirectional=True, batch_first=True, dropout=dropout)
                self.dropout = nn.Dropout(dropout)
                self.hidden2tag = nn.Linear(hidden_dim * 2, tag_count)
                self.crf = CRF(tag_count, batch_first=True)

            def forward(self, words, lemmas, pos_tags, tags=None, mask=None):
                word_emb = self.word_embedding(words)
                lemma_emb = self.lemma_embedding(lemmas)
                pos_emb = self.pos_embedding(pos_tags)

                combined_emb = torch.cat([word_emb, lemma_emb, pos_emb], dim=2)
                lstm_out, _ = self.lstm(combined_emb)
                lstm_out = self.dropout(lstm_out)
                logits = self.hidden2tag(lstm_out)

                if tags is not None:
                    loss = -self.crf(logits, tags, mask=mask, reduction="mean")
                    return {"loss": loss}
                else:
                    predictions = self.crf.decode(logits, mask=mask)
                    return {"predictions": predictions}

        self.model = BiLSTMCRFWithFeatures(
            word_vocab_size=len(self.processor.word2idx),
            lemma_vocab_size=len(self.processor.lemma2idx),
            pos_vocab_size=len(self.processor.pos2idx),
            tag_count=len(self.processor.tag2idx),
            word_emb_dim=self.word_emb_dim,
            lemma_emb_dim=self.lemma_emb_dim,
            pos_emb_dim=self.pos_emb_dim,
            hidden_dim=self.hidden_dim,
            dropout=self.dropout,
            word_pad_idx=self.processor.word2idx["<PAD>"],
            lemma_pad_idx=self.processor.lemma2idx["<PAD>"],
            pos_pad_idx=self.processor.pos2idx["<PAD>"]
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def create_data_loaders(self, splits):
        """Create data loaders"""
        train_dataset = NERDatasetFull(*splits['train'])
        val_dataset = NERDatasetFull(*splits['val'])
        test_dataset = NERDatasetFull(*splits['test'])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size)

        return train_loader, val_loader, test_loader

    def train(self, train_loader, val_loader, epochs=50, patience=7):
        """Train the model with early stopping"""
        print("Training BiLSTM-CRF with Full Features Model...")

        early_stopping = EarlyStopping(patience=patience, min_delta=0.0001)

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                words, lemmas, pos_tags, tags = batch
                words = words.to(self.device)
                lemmas = lemmas.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)
                mask = (words != self.processor.word2idx["<PAD>"]).to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(words, lemmas, pos_tags, tags=tags, mask=mask)
                loss = outputs["loss"]
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                train_loss += loss.item()

            # Validation
            entity_f1, token_f1 = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss/len(train_loader):.4f}, "
                  f"Entity F1={entity_f1:.4f}, Token F1={token_f1:.4f}")

            # Track best scores and save best model
            if entity_f1 > self.best_entity_f1:
                self.best_entity_f1 = entity_f1
                self.best_token_f1 = token_f1
                torch.save(self.model.state_dict(), "best_bilstm_crf_features.pth")

            # Early stopping check
            early_stopping(entity_f1, self.model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

        # Restore best weights if early stopping was used
        if early_stopping.restore_best_weights and early_stopping.best_weights is not None:
            self.model.load_state_dict(early_stopping.best_weights)

    def evaluate(self, data_loader):
        """Evaluate the model"""
        self.model.eval()
        y_true_sequences, y_pred_sequences = [], []

        with torch.no_grad():
            for batch in data_loader:
                words, lemmas, pos_tags, tags = batch
                words = words.to(self.device)
                lemmas = lemmas.to(self.device)
                pos_tags = pos_tags.to(self.device)
                tags = tags.to(self.device)
                mask = (words != self.processor.word2idx["<PAD>"]).to(self.device)

                outputs = self.model(words, lemmas, pos_tags, mask=mask)
                predictions = outputs["predictions"]

                for i, pred_seq in enumerate(predictions):
                    true_seq = tags[i][mask[i]].cpu().numpy()
                    y_true_sequences.append(true_seq)
                    y_pred_sequences.append(pred_seq)

        return NEREvaluator.calculate_metrics(y_true_sequences, y_pred_sequences, self.processor.idx2tag)

    def test(self, test_loader):
        """Test the model with best weights"""
        self.model.load_state_dict(torch.load("best_bilstm_crf_features.pth"))
        return self.evaluate(test_loader)



In [12]:
def main():
    """Main execution function with comprehensive model comparison and early stopping"""

    # Initialize data processor
    print("Loading and preprocessing data...")
    processor = NERDataProcessor("combined_dataset.csv")
    processor.load_data()
    processor.build_vocabularies()
    X_words_pad, X_lemmas_pad, X_pos_pad, y_ner_pad = processor.prepare_sequences()
    splits = processor.create_splits(X_words_pad, X_lemmas_pad, X_pos_pad, y_ner_pad)

    # Convert to tensors
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for split_name in ['train', 'val', 'test']:
        Xw, Xl, Xp, yn = splits[split_name]
        splits[split_name] = (
            torch.tensor(Xw, dtype=torch.long).to(device),
            torch.tensor(Xl, dtype=torch.long).to(device),
            torch.tensor(Xp, dtype=torch.long).to(device),
            torch.tensor(yn, dtype=torch.long).to(device)
        )

    # Initialize all 6 models
    models = {
        "BiLSTM Simple": BiLSTMSimpleModel(processor),
        "BiLSTM + POS": BiLSTMPOSModel(processor),
        "BiLSTM + Full Features": BiLSTMFeaturesModel(processor),
        "BiLSTM-CRF Simple": BiLSTMCRFSimpleModel(processor),
        "BiLSTM-CRF + POS": BiLSTMCRFPOSModel(processor),
        "BiLSTM-CRF + Full Features": BiLSTMCRFFeaturesModel(processor)
    }

    # Training parameters with early stopping
    epochs = 50  # Increased since we have early stopping
    patience = 5  # Stop if no improvement for 7 epochs

    # Results storage
    results = []

    # Train and evaluate each model
    for model_name, model in models.items():
        print(f"\n{'='*70}")
        print(f"Processing {model_name}")
        print('='*70)

        # Build model and create data loaders
        model.build_model()
        train_loader, val_loader, test_loader = model.create_data_loaders(splits)

        # Train the model with early stopping
        model.train(train_loader, val_loader, epochs=epochs, patience=patience)

        # Test the model
        test_entity_f1, test_token_f1 = model.test(test_loader)

        # Store results
        results.append({
            'Model': model_name,
            'Architecture': 'BiLSTM-CRF' if 'CRF' in model_name else 'BiLSTM',
            'Features': 'Words Only' if 'Simple' in model_name else ('Words + POS' if '+ POS' in model_name else 'Words + Lemmas + POS'),
            'Entity-level F1': f"{test_entity_f1:.4f}",
            'Token-level F1': f"{test_token_f1:.4f}"
        })

        print(f"\n{model_name} Final Results:")
        print(f"Entity-level F1: {test_entity_f1:.4f}")
        print(f"Token-level F1: {test_token_f1:.4f}")

    results_df = pd.DataFrame(results)

    print("\n" + "="*90)
    print("COMPREHENSIVE MODEL COMPARISON TABLE (WITH EARLY STOPPING)")
    print("="*90)
    print(results_df.to_string(index=False))

    print("\n" + "="*90)
    print("FEATURE COMPARISON MATRIX (Entity-level F1)")
    print("="*90)

    feature_comparison = results_df.pivot_table(
        index='Features',
        columns='Architecture',
        values='Entity-level F1',
        aggfunc='first'
    )
    print(feature_comparison)


    results_df.to_csv("complete_ner_model_comparison_early_stopping.csv", index=False)
    feature_comparison.to_csv("feature_comparison_matrix_early_stopping.csv")

    print(f"\nResults saved to:")
    print(f"- complete_ner_model_comparison_early_stopping.csv")
    print(f"- feature_comparison_matrix_early_stopping.csv")

    return results_df, feature_comparison


# Execute main function
if __name__ == "__main__":
    results_table, feature_matrix = main()


Loading and preprocessing data...

Processing BiLSTM Simple
Training BiLSTM Simple Model...
Epoch 1/50: Loss=0.3108, Entity F1=0.6716, Token F1=0.9609
Epoch 2/50: Loss=0.1144, Entity F1=0.7660, Token F1=0.9712
Epoch 3/50: Loss=0.0720, Entity F1=0.7974, Token F1=0.9745
Epoch 4/50: Loss=0.0495, Entity F1=0.8130, Token F1=0.9753
Epoch 5/50: Loss=0.0356, Entity F1=0.8198, Token F1=0.9756
Epoch 6/50: Loss=0.0265, Entity F1=0.8253, Token F1=0.9765
Epoch 7/50: Loss=0.0206, Entity F1=0.8277, Token F1=0.9768
Epoch 8/50: Loss=0.0161, Entity F1=0.8255, Token F1=0.9763
EarlyStopping counter: 1 out of 5
Epoch 9/50: Loss=0.0132, Entity F1=0.8221, Token F1=0.9758
EarlyStopping counter: 2 out of 5
Epoch 10/50: Loss=0.0112, Entity F1=0.8217, Token F1=0.9761
EarlyStopping counter: 3 out of 5
Epoch 11/50: Loss=0.0096, Entity F1=0.8175, Token F1=0.9757
EarlyStopping counter: 4 out of 5
Epoch 12/50: Loss=0.0084, Entity F1=0.8201, Token F1=0.9761
EarlyStopping counter: 5 out of 5
Early stopping triggered at